# 01 — RAG Pipeline Basics

**Vai trò:** Pipeline Engineer · **Task:** S3-PE-07 (Yêu cầu 9.3)

Notebook này minh hoạ `RAGPipeline` (S1-PE-02, S2-PE-01, S3-PE-03) — class điều phối trung tâm chạy trọn vẹn luồng RAG **thật**: `index_document()` (Load → Chunk → Embed → Store) rồi `query()` (Embed câu hỏi → Retrieve → Build Prompt → Generate với LLM cục bộ qua OLLAMA), đúng pseudocode design.md §2.7.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loader import DocumentLoader
from src.data.chunker import TextChunker
from src.embeddings.embedding_model import OllamaEmbeddingModel
from src.embeddings.vector_store import ChromaVectorStore
from src.generation.llm_client import OllamaClient
from src.generation.prompt_builder import PromptBuilder
from src.models import ChunkStrategy
from src.pipeline.rag_pipeline import RAGPipeline

print(f"Project root: {PROJECT_ROOT}")

Project root: D:\lh222k\AI-Research-Assistant-with-RAG


## 1. Chuẩn bị tài liệu mẫu

Tái sử dụng tài liệu mẫu đã tạo ở [`01_document_loading.ipynb`](../data_engineer/01_document_loading.ipynb) (`data/raw/sample_rag_overview.txt`). Việc ghi file là **idempotent** — chạy lại notebook không tạo bản sao, đảm bảo notebook chạy độc lập từ đầu đến cuối (Yêu cầu 9.1).

In [2]:
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
sample_path = RAW_DIR / "sample_rag_overview.txt"

if not sample_path.exists():
    sample_path.write_text(
        "Retrieval-Augmented Generation (RAG) la kien truc ket hop "
        "retrieval va generation. He thong tim cac doan van ban lien "
        "quan tu kho du lieu rieng truoc khi yeu cau LLM sinh cau tra "
        "loi, giup giam hien tuong ao giac (hallucination) va bam sat "
        "nguon tai lieu thuc te.\n\nQuy trinh RAG gom hai giai doan: "
        "indexing (tai tai lieu, chia nho thanh chunk, tao embedding, "
        "luu vao vector store) va querying (embed cau hoi, tim chunk "
        "lien quan nhat, ghep prompt, goi LLM sinh cau tra loi).",
        encoding="utf-8",
    )
    print(f"Đã tạo: {sample_path.relative_to(PROJECT_ROOT)}")
else:
    print(f"Đã tồn tại: {sample_path.relative_to(PROJECT_ROOT)}")

Đã tồn tại: data\raw\sample_rag_overview.txt


## 2. Khởi tạo `RAGPipeline` với các thành phần thật

Notebook dùng `ChromaVectorStore` ở chế độ **in-memory** (`persist_dir=None`) để không ảnh hưởng tới collection persistent của dashboard — đây chính là chế độ song song mà `BaseVectorStore` phải hỗ trợ (Yêu cầu 4.5, 4.6).

In [3]:
pipeline = RAGPipeline(
    loader=DocumentLoader(),
    chunker=TextChunker(strategy=ChunkStrategy.RECURSIVE, chunk_size=300, chunk_overlap=50),
    embedding_model=OllamaEmbeddingModel(model_name="nomic-embed-text"),
    vector_store=ChromaVectorStore(collection_name="notebook_rag_basics", persist_dir=None),
    llm_client=OllamaClient(model_name="llama3", max_tokens=200),
    prompt_builder=PromptBuilder(),
    top_k=3,
)
print("RAGPipeline đã sẵn sàng — top_k =", pipeline.top_k)

RAGPipeline đã sẵn sàng — top_k = 3


## 3. Kiểm tra OLLAMA khả dụng trước khi chạy luồng đầy đủ

`is_available()` không bao giờ ném exception (Yêu cầu 5.2) — notebook dùng kết quả này để quyết định có chạy `index_document()`/`query()` thật hay chỉ hiển thị hướng dẫn, đảm bảo chạy hết từ đầu đến cuối không lỗi (Yêu cầu 9.3) dù OLLAMA đã sẵn sàng hay chưa.

In [4]:
available = pipeline.llm_client.is_available()
print(f"OLLAMA khả dụng tại {pipeline.llm_client.base_url}: {available}")

if not available:
    print(
        "\n⚠️  OLLAMA chưa sẵn sàng. Hãy chạy 'ollama serve' và "
        "'ollama pull llama3' / 'ollama pull nomic-embed-text' rồi chạy lại notebook."
    )

OLLAMA khả dụng tại http://localhost:11434: True


## 4. Indexing — `index_document()` (Load → Chunk → Embed → Store)

Theo pseudocode design.md §2.7: tải tài liệu, chia chunk, tạo embedding cho từng chunk (loop invariant `len(vectors) == i`), rồi lưu vào vector store. Kết quả là một `IndexingResult` với `success=True` và `num_chunks` thật.

In [5]:
indexing_result = None
if available:
    indexing_result = pipeline.index_document(str(sample_path))
    print(indexing_result)
    assert indexing_result.success, indexing_result.error_message
    assert indexing_result.num_chunks >= 1
else:
    print("Bỏ qua indexing thật — OLLAMA không khả dụng (xem mục 3).")

IndexingResult(doc_id='4c69f82b6454088f', num_chunks=5, collection_name='notebook_rag_basics', success=True, error_message=None)


## 5. Querying — `query()` (Embed câu hỏi → Retrieve → Build Prompt → Generate)

`RAGResponse` trả về chứa `answer` (sinh thật từ LLM cục bộ), `contexts` (các `ScoredChunk` đã truy xuất) và `latency_ms` đo trọn vẹn 4 bước (Yêu cầu 7.3, 7.4).

In [6]:
response = None
if available and indexing_result is not None and indexing_result.success:
    question = "RAG la gi va gom nhung giai doan nao?"
    response = pipeline.query(question)

    print(f"Câu hỏi : {response.question}")
    print(f"Model   : {response.model_name}")
    print(f"Latency : {response.latency_ms:.1f} ms")
    print(f"Trả lời : {response.answer[:400]}")
    print(f"\nSố context truy xuất: {len(response.contexts)}")
    for sc in response.contexts:
        print(f"  #{sc.rank} score={sc.score:.3f} doc_id={sc.chunk.doc_id} vị trí=[{sc.chunk.start_index}:{sc.chunk.end_index}]")
else:
    print("Bỏ qua query thật — cần indexing thành công và OLLAMA khả dụng trước.")

Câu hỏi : RAG la gi va gom nhung giai doan nao?
Model   : llama3
Latency : 57462.5 ms
Trả lời : Based on the provided context, I can answer your question.

According to Đoạn 3 (score=0.834), RAG (Reinforcement Augmented Generation) has two main stages: indexing and querying.

Indexing stage includes:

* Indexing data
* Chunking data into smaller chunks
* Creating embeddings and storing them in a vector store

Querying stage includes:

* Embedding the query
* Finding the most relevant chunks 

Số context truy xuất: 3
  #1 score=0.861 doc_id=4c69f82b6454088f vị trí=[147:425]
  #2 score=0.840 doc_id=4c69f82b6454088f vị trí=[955:1255]
  #3 score=0.834 doc_id=4c69f82b6454088f vị trí=[425:673]


## 6. Xác minh Property 10 — Indexing round-trip

*Với mọi tài liệu hợp lệ, sau khi `index_document()` thành công, một câu hỏi liên quan phải trả về ít nhất một `ScoredChunk` từ đúng tài liệu đó trong `RAGResponse.contexts`* (design.md Phần 3, Property 10 — Validates Yêu cầu 7.1, 7.2).

In [7]:
if response is not None:
    assert response.answer, "answer không được rỗng (Yêu cầu 7.3)"
    assert response.latency_ms > 0, "latency_ms phải > 0 (Yêu cầu 7.4)"
    assert len(response.contexts) >= 1, "Property 10: phải truy xuất được ít nhất 1 context"
    assert all(sc.chunk.doc_id == indexing_result.doc_id for sc in response.contexts), (
        "Property 10: context phải đến từ đúng tài liệu vừa index"
    )
    print("Property 10 OK — tài liệu vừa index có thể truy xuất lại đúng qua query().")
else:
    print("Bỏ qua xác minh Property 10 — chưa có RAGResponse thật (xem mục 3).")

Property 10 OK — tài liệu vừa index có thể truy xuất lại đúng qua query().


## 7. Tổng kết

- `RAGPipeline` điều phối trọn vẹn luồng RAG — từ `index_document()` (Load → Chunk → Embed → Store) đến `query()` (Embed → Retrieve → Build Prompt → Generate) — đúng pseudocode design.md §2.7, dùng chung giữa notebook và Streamlit dashboard (Yêu cầu 7).
- `RAGResponse.latency_ms` đo trọn vẹn thời gian xử lý một câu hỏi; `RAGResponse.contexts` chứa các `ScoredChunk` đã truy xuất kèm `score`/`rank`/vị trí gốc — đây chính là dữ liệu hiển thị ở trang **Retrieval Debug** của dashboard.
- **Property 10** (indexing round-trip) được xác minh trực tiếp: tài liệu vừa index luôn có thể truy xuất lại qua một câu hỏi liên quan.
- Notebook tiếp theo [`02_retrieval_strategies.ipynb`](02_retrieval_strategies.ipynb) đào sâu vào bước **Retrieve** (so sánh dense/sparse/hybrid), và [`03_prompt_engineering.ipynb`](03_prompt_engineering.ipynb) đào sâu vào bước **Build Prompt**.